# Assignment 3b — Neural Network from Scratch (template)

Fill in the function stubs. Shape hints are in the comments — the goal is a working 2→8→1 network trained with manual backprop on `make_moons`.

**[Guide: Neural Network from Scratch](https://osu-car-msl.github.io/lab-setup-guide/assignments/neural-network-from-scratch/)**


## Part 0 — Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier


## Part 1 — Decision Tree Baseline

Generate `make_moons` and visualize it. Then train a decision tree and plot its decision boundary — you'll compare your neural net against this at the end.


In [ ]:
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)

plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], c="steelblue", label="Class 0", alpha=0.6)
plt.scatter(X[y == 1, 0], X[y == 1, 1], c="coral", label="Class 1", alpha=0.6)
plt.title("make_moons Dataset (n=500, noise=0.2)")
plt.xlabel("x₁"); plt.ylabel("x₂")
plt.legend(); plt.show()


In [ ]:
def plot_decision_boundary(model_predict, X, y, title, ax=None):
    """Plot the decision boundary of a classifier."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model_predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="steelblue", label="Class 0", alpha=0.6)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="coral", label="Class 1", alpha=0.6)
    ax.set_title(title); ax.set_xlabel("x₁"); ax.set_ylabel("x₂"); ax.legend()

dt = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X, y)
plot_decision_boundary(dt.predict, X, y, "Decision Tree (max_depth=5)")
plt.show()


### Checkpoint

- [ ] `make_moons` scatter plotted
- [ ] Decision tree boundary plotted (note the staircase shape)


## Part 2 — Building Blocks


In [ ]:
def sigmoid(z):
    """Sigmoid activation: 1 / (1 + e^-z)."""
    # TODO
    return ...


def sigmoid_derivative(a):
    """Derivative of sigmoid given the sigmoid output a = sigmoid(z)."""
    # TODO: return a * (1 - a)
    return ...


# Sanity check
z = np.array([-5, -1, 0, 1, 5])
print(f"sigmoid({z}) = {sigmoid(z).round(4) if callable(sigmoid) else '...'}")
# Expected: [0.0067, 0.2689, 0.5, 0.7311, 0.9933]


## Part 3 — Forward Pass

Architecture:

- Input: 2 features (`x₁`, `x₂`)
- Hidden: 8 neurons, sigmoid
- Output: 1 neuron, sigmoid


In [ ]:
np.random.seed(42)

W1 = np.random.randn(2, 8) * 0.5    # (2 inputs, 8 hidden)
b1 = np.zeros((1, 8))                # (1, 8)
W2 = np.random.randn(8, 1) * 0.5    # (8 hidden, 1 output)
b2 = np.zeros((1, 1))                # (1, 1)

print(f"W1: {W1.shape}, b1: {b1.shape}, W2: {W2.shape}, b2: {b2.shape}")


In [ ]:
def forward(X, W1, b1, W2, b2):
    """
    X  shape: (n, 2)
    Z1 shape: (n, 8)  — hidden pre-activation
    A1 shape: (n, 8)  — hidden post-activation
    Z2 shape: (n, 1)  — output pre-activation
    A2 shape: (n, 1)  — output (prediction)
    """
    # TODO: Z1 = X @ W1 + b1
    # TODO: A1 = sigmoid(Z1)
    # TODO: Z2 = A1 @ W2 + b2
    # TODO: A2 = sigmoid(Z2)
    Z1 = A1 = Z2 = A2 = ...
    return Z1, A1, Z2, A2


# Test on 5 samples
Z1, A1, Z2, A2 = forward(X[:5], W1, b1, W2, b2)
print(f"Predictions: {A2.flatten().round(4) if hasattr(A2, 'flatten') else '...'}")
print(f"Actual:      {y[:5]}")


### Checkpoint

- [ ] `sigmoid` and `sigmoid_derivative` pass the sanity check
- [ ] `forward` runs on a sample batch without errors


## Part 4 — Loss and Backpropagation


In [ ]:
def compute_loss(y_true, y_pred):
    """Binary cross-entropy loss."""
    # TODO: clip y_pred to avoid log(0), then compute BCE
    y_pred = np.clip(y_pred, 1e-8, 1 - 1e-8)
    loss = ...
    return loss


In [ ]:
def backward(X, y_true, Z1, A1, Z2, A2, W1, b1, W2, b2):
    """
    Return gradients: dW1, db1, dW2, db2

    Shapes:
        dZ2: (n, 1)      dW2: (8, 1)   db2: (1, 1)
        dA1: (n, 8)
        dZ1: (n, 8)      dW1: (2, 8)   db1: (1, 8)
    """
    n = X.shape[0]
    y_true = y_true.reshape(-1, 1)

    # TODO: dZ2 = A2 - y_true
    dZ2 = ...

    # TODO: dW2 = (A1.T @ dZ2) / n
    # TODO: db2 = np.sum(dZ2, axis=0, keepdims=True) / n
    dW2 = db2 = ...

    # TODO: dA1 = dZ2 @ W2.T
    # TODO: dZ1 = dA1 * sigmoid_derivative(A1)
    dZ1 = ...

    # TODO: dW1 = (X.T @ dZ1) / n
    # TODO: db1 = np.sum(dZ1, axis=0, keepdims=True) / n
    dW1 = db1 = ...

    return dW1, db1, dW2, db2


## Part 5 — Training Loop


In [ ]:
learning_rate = 0.5
epochs = 1000

np.random.seed(42)
W1 = np.random.randn(2, 8) * 0.5
b1 = np.zeros((1, 8))
W2 = np.random.randn(8, 1) * 0.5
b2 = np.zeros((1, 1))

losses = []

for epoch in range(epochs):
    # TODO: forward pass
    Z1, A1, Z2, A2 = forward(X, W1, b1, W2, b2)

    # TODO: loss
    loss = compute_loss(y, A2)
    losses.append(loss)

    # TODO: backward + weight updates
    dW1, db1, dW2, db2 = backward(X, y, Z1, A1, Z2, A2, W1, b1, W2, b2)
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    if epoch % 100 == 0:
        accuracy = np.mean((A2.flatten() > 0.5) == y)
        print(f"Epoch {epoch:4d} | Loss: {loss:.4f} | Accuracy: {accuracy:.3f}")

print(f"\nFinal — Loss: {losses[-1]:.4f} | Accuracy: {np.mean((A2.flatten() > 0.5) == y):.3f}")


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(losses)
plt.title("Training Loss Over Time")
plt.xlabel("Epoch"); plt.ylabel("Binary Cross-Entropy Loss")
plt.grid(True, alpha=0.3); plt.show()


In [ ]:
def nn_predict(X_input):
    _, _, _, A2 = forward(X_input, W1, b1, W2, b2)
    return (A2.flatten() > 0.5).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_decision_boundary(dt.predict, X, y, "Decision Tree (max_depth=5)", ax=axes[0])
plot_decision_boundary(nn_predict, X, y, "Neural Network (2→8→1)", ax=axes[1])
plt.tight_layout(); plt.show()


### Hyperparameter sweep

Wrap the training loop in a function and sweep at least 3 configurations. Suggested sweep:

| Hidden size | Learning rate |
|:-----------:|:-------------:|
| 4 | 0.5 |
| 8 | 0.5 |
| 16 | 0.5 |
| 8 | 0.1 |
| 8 | 1.0 |


In [ ]:
def train(hidden_size: int, learning_rate: float, epochs: int = 1000, seed: int = 42):
    """Return ((W1, b1, W2, b2), losses)."""
    rng = np.random.default_rng(seed)
    W1 = rng.standard_normal((2, hidden_size)) * 0.5
    b1 = np.zeros((1, hidden_size))
    W2 = rng.standard_normal((hidden_size, 1)) * 0.5
    b2 = np.zeros((1, 1))

    losses = []
    for _ in range(epochs):
        Z1, A1, Z2, A2 = forward(X, W1, b1, W2, b2)
        losses.append(compute_loss(y, A2))
        dW1, db1, dW2, db2 = backward(X, y, Z1, A1, Z2, A2, W1, b1, W2, b2)
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
    return (W1, b1, W2, b2), losses


# TODO: run at least 3 configs and save decision boundary plots for each
# configs = [(4, 0.5), (8, 0.5), (16, 0.5)]
# results = {cfg: train(*cfg) for cfg in configs}


### Checkpoint

- [ ] Loss decreases smoothly over epochs
- [ ] NN decision boundary is smooth and follows the moons (vs tree's staircase)
- [ ] At least 3 hyperparameter experiments run


## Part 6 — Sanity check against sklearn

`MLPClassifier` does everything you just built in 4 lines. Compare the accuracy.


In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(8,), activation="logistic", max_iter=1000, random_state=42
)
mlp.fit(X, y)
print(f"sklearn MLP accuracy: {mlp.score(X, y):.3f}")


## Part 7 — Publish

Convert this notebook to a blog post. Suggested categories: `[deep-learning, python, reflection]`. Lead with the side-by-side decision boundary comparison.

### Final deliverables

- [ ] Decision tree baseline plot
- [ ] Working `forward()` and `backward()` (loss decreases)
- [ ] Loss curve plot
- [ ] Side-by-side decision boundary comparison
- [ ] Results from at least 3 hyperparameter experiments
- [ ] Three reflection answers (guide Parts 2, 3, 6)
- [ ] Blog post URL
